In [1]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn

In [2]:
df = pd.read_csv(r"fmnist_small.csv")
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,9,0,0,0,0,0,0,0,0,0,...,0,7,0,50,205,196,213,165,0,0
1,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,1,0,0,0,...,142,142,142,21,0,3,0,0,0,0
3,8,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,8,0,0,0,0,0,0,0,0,0,...,213,203,174,151,188,10,0,0,0,0


In [3]:
x = df.iloc[:, 1:]
y = df.iloc[:,0]

In [4]:
x

,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,pixel10,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,0,0,0,0,0,0,0,0,0,0,...,0,7,0,50,205,196,213,165,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,1,0,0,0,0,...,142,142,142,21,0,3,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,213,203,174,151,188,10,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5995,0,0,0,0,0,0,0,0,0,1,...,69,12,0,0,0,0,0,0,0,0
5996,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5997,0,0,0,0,0,0,0,0,0,0,...,39,47,2,0,0,29,0,0,0,0
5998,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0


In [5]:
y

0       9
1       7
2       0
3       8
4       8
       ..
5995    1
5996    5
5997    8
5998    4
5999    8
Name: label, Length: 6000, dtype: int64

In [6]:
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2,random_state=42)

In [7]:
x_train = x_train/255.0
x_test = x_test/255.0

In [8]:
class CustomDataset(Dataset):

    def __init__(self, features, labels):
        self.features = torch.tensor(features.values, dtype=torch.float32)
        self.labels = torch.tensor(labels.values, dtype=torch.long)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        return self.features[index], self.labels[index]

In [9]:
train_dataset = CustomDataset(x_train, y_train)
test_dataset = CustomDataset(x_test, y_test)

In [10]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [11]:
class MyNN(nn.Module):

    def __init__(self, num_features):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(num_features, 128),
            nn.ReLU(),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 10),
            nn.Softmax()
        )

    def forward(self, features):
        return self.network(features)    


In [12]:
learning_rate = 0.1
epoch = 100
loss_function = nn.CrossEntropyLoss()


In [13]:
model = MyNN(x_train.shape[1])

optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

In [14]:
for epoch in range(epoch):
    total_loss = 0
    for features, lables in train_loader:
        y_pred = model(features)

        loss = loss_function(y_pred, lables)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss/len(train_loader)    

    print(epoch+1, avg_loss)    

c:\Desktop\PyTorch\.venv\Lib\site-packages\torch\nn\modules\module.py:1739: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)


1 2.2904860226313275
2 2.174445631504059
3 1.9586823550860086
4 1.8677718313535054
5 1.8390889811515807
6 1.81542591492335
7 1.7894420607884725
8 1.774185398419698
9 1.7647102983792622
10 1.7566797908147176
11 1.7539827791849771
12 1.7471617166201274
13 1.7430137292544048
14 1.738257159392039
15 1.737310783068339
16 1.733230751355489
17 1.7309920811653137
18 1.7291066948572795
19 1.7279929169019064
20 1.7240637016296387
21 1.7245368369420369
22 1.72259699344635
23 1.7199763186772665
24 1.7194211943944295
25 1.7184039076169333
26 1.7172405751546225
27 1.7149713762601217
28 1.7136034417152404
29 1.7123640433947245
30 1.7117261536916097
31 1.7117412090301514
32 1.7113216161727904
33 1.7098055092493694
34 1.7090699156125386
35 1.7087995958328248
36 1.709465627670288
37 1.7077154819170635
38 1.7052999329566956
39 1.7052361297607421
40 1.7043464692433674
41 1.704894287586212
42 1.7037877178192138
43 1.7038758452733358
44 1.703390531539917
45 1.7027033869425456
46 1.7021710228919984
47 1.7009

In [16]:
model.eval()

MyNN(
  (network): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=10, bias=True)
    (5): Softmax(dim=None)
  )
)

In [17]:
total = 0
correct = 0


with torch.no_grad():
    for features, labels in train_loader:
        y_pred = model(features)
        _, predicted = torch.max(y_pred, 1)
        total+=lables.size(0)
        correct+=(predicted==labels).sum().item()
accuracy = correct/total
print(accuracy)        

0.7710416666666666


c:\Desktop\PyTorch\.venv\Lib\site-packages\torch\nn\modules\module.py:1739: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)
